# 01 — Data Acquisition

This notebook retrieves non-metallic inorganic oxide data from the Materials Project API, applies the project filters, creates reduced formulas, and saves the raw project dataset.

The Materials Project API key is loaded from a local `.env` file and is never stored in this notebook.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

# Search the current folder and parent folders for the local .env file
env_path = find_dotenv(usecwd=True)

if not env_path:
    raise FileNotFoundError(
        "No .env file was found. Create one in the repository root "
        "using .env.example as the template."
    )

load_dotenv(env_path)

mp_api_key = os.getenv("MP_API_KEY")

if not mp_api_key:
    raise ValueError(
        "MP_API_KEY is missing from the local .env file."
    )

print("Materials Project API key loaded successfully.")
print("The key itself is not displayed.")

Materials Project API key loaded successfully.
The key itself is not displayed.


In [2]:
from mp_api.client import MPRester

test_fields = [
    "material_id",
    "formula_pretty",
    "band_gap",
    "is_metal",
    "formation_energy_per_atom",
    "energy_above_hull",
    "density",
    "volume",
    "nsites",
    "nelements",
    "elements",
    "symmetry"
]

with MPRester(mp_api_key) as mpr:
    test_docs = mpr.materials.summary.search(
        material_ids=["mp-149", "mp-13"],
        fields=test_fields
    )

print(f"Test records returned: {len(test_docs)}")

for doc in test_docs:
    print(
        doc.material_id,
        doc.formula_pretty,
        doc.band_gap,
        doc.symmetry.crystal_system
    )

Retrieving SummaryDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Test records returned: 2
mp-149 Si 0.6102999999999996 Cubic
mp-13 Fe 0.0 Cubic


In [3]:
from pathlib import Path
import warnings

import pandas as pd
from mp_api.client import MPRester
from pymatgen.core import Composition


# ------------------------------------------------------------
# 1. Locate the repository and output directory
# ------------------------------------------------------------
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    data_dir
    / "zhao_nonmetal_oxide_bandgap_project.csv"
)


# ------------------------------------------------------------
# 2. Query fields
# ------------------------------------------------------------
query_fields = [
    "material_id",
    "formula_pretty",
    "band_gap",
    "is_metal",
    "formation_energy_per_atom",
    "energy_above_hull",
    "density",
    "volume",
    "nsites",
    "nelements",
    "elements",
    "symmetry"
]


# ------------------------------------------------------------
# 3. Download non-metallic oxygen-containing materials
# ------------------------------------------------------------
print(
    "Downloading non-metallic oxygen-containing compounds..."
)

with MPRester(mp_api_key) as mpr:
    docs = mpr.materials.summary.search(
        elements=["O"],
        num_elements=(2, 4),
        is_metal=False,
        band_gap=(0.1, 20.0),
        fields=query_fields,
        chunk_size=1000
    )

print(
    f"Records returned by API: {len(docs):,}"
)


# ------------------------------------------------------------
# 4. Apply project filters and construct clean records
# ------------------------------------------------------------
rows = []

for doc in docs:

    if doc.formula_pretty is None:
        continue

    element_symbols = [
        str(element)
        for element in doc.elements
    ]

    non_oxygen_elements = [
        element
        for element in element_symbols
        if element != "O"
    ]

    # Oxygen plus one to three non-oxygen elements
    if len(non_oxygen_elements) not in (1, 2, 3):
        continue

    if doc.band_gap is None or doc.band_gap <= 0.1:
        continue

    if doc.is_metal is True:
        continue

    symmetry = getattr(
        doc,
        "symmetry",
        None
    )

    crystal_system_object = getattr(
        symmetry,
        "crystal_system",
        None
    )

    if crystal_system_object is None:
        continue

    if hasattr(crystal_system_object, "value"):
        crystal_system = str(
            crystal_system_object.value
        ).capitalize()
    else:
        crystal_system = (
            str(crystal_system_object)
            .split(".")[-1]
            .capitalize()
        )

    formula = str(doc.formula_pretty)

    # Suppress only pymatgen's harmless helium
    # electronegativity warning.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=(
                "No Pauling electronegativity for He.*"
            ),
            category=UserWarning
        )

        reduced_formula = (
            Composition(formula).reduced_formula
        )

    rows.append({
        "material_id": str(doc.material_id),
        "formula": formula,
        "reduced_formula": reduced_formula,
        "band_gap": doc.band_gap,
        "is_metal": doc.is_metal,
        "formation_energy_per_atom":
            doc.formation_energy_per_atom,
        "energy_above_hull":
            doc.energy_above_hull,
        "density": doc.density,
        "volume": doc.volume,
        "nsites": doc.nsites,
        "nelements": doc.nelements,
        "elements": ",".join(element_symbols),
        "crystal_system": crystal_system
    })


# ------------------------------------------------------------
# 5. Create and save the final raw project dataset
# ------------------------------------------------------------
column_order = [
    "material_id",
    "formula",
    "reduced_formula",
    "band_gap",
    "is_metal",
    "formation_energy_per_atom",
    "energy_above_hull",
    "density",
    "volume",
    "nsites",
    "nelements",
    "elements",
    "crystal_system"
]

df_raw = (
    pd.DataFrame(rows)
    .drop_duplicates(
        subset="material_id"
    )
    .reset_index(drop=True)
)

df_raw = df_raw[column_order]

df_raw.to_csv(
    output_path,
    index=False
)


# ------------------------------------------------------------
# 6. Report acquisition results
# ------------------------------------------------------------
print("\nData acquisition complete.")
print("Dataset shape:", df_raw.shape)
print(
    "Unique reduced formulas:",
    df_raw["reduced_formula"].nunique()
)
print("Saved to:", output_path.resolve())

print("\nCrystal-system counts:")
print(
    df_raw["crystal_system"].value_counts()
)

display(df_raw.head())

Retrieving SummaryDoc documents:   0%|          | 0/40079 [00:00<?, ?it/s]

Records returned by API: 40,079

Data acquisition complete.
Dataset shape: (40078, 13)
Unique reduced formulas: 22816
Saved to: C:\Users\24250\Desktop\zhao-oxide-bandgap-project\data\zhao_nonmetal_oxide_bandgap_project.csv

Crystal-system counts:
crystal_system
Monoclinic      14447
Triclinic        9095
Orthorhombic     8160
Trigonal         2796
Tetragonal       2727
Cubic            1782
Hexagonal        1071
Name: count, dtype: int64


,material_id,formula,reduced_formula,band_gap,is_metal,formation_energy_per_atom,energy_above_hull,density,volume,nsites,nelements,elements,crystal_system
0,mp-3346792,HCNO,HCNO,5.2381,False,-0.680053,0.000000,1.823257,235.109820,24,4,"C,H,N,O",Monoclinic
1,mp-1523737,NClO,NClO,2.0666,False,0.090138,0.159467,1.812475,239.887179,12,3,"Cl,N,O",Monoclinic
2,mp-1567687,V3OF7,V3OF7,1.3813,False,-3.022540,0.096936,3.215234,311.748264,22,3,"F,O,V",Monoclinic
3,mp-1568124,V3OF7,V3OF7,1.3700,False,-3.014306,0.105170,3.227839,310.530861,22,3,"F,O,V",Triclinic
4,mp-1568829,Fe4OF6,Fe4OF6,2.6285,False,-2.323263,0.065270,4.420859,265.461720,22,3,"F,Fe,O",Triclinic
